<a href="https://colab.research.google.com/github/Yusuf-kadri/Yusuf-kadri---AnalyticsEngineering/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

In [4]:
# 1. Look at the data
from IPython.display import display

# Let's inspect the tables stored in tables dictionary
for name, df in tables.items():
    print(f"=== Table: {name} ({df.shape[0]} rows, {df.shape[1]} columns) ===")

    # Display first few rows
    display(df.head())

    # Check data types and missing values
    print("\n--- Info & Nulls ---")
    print(df.info())

    # Check for duplicates on the primary keys
    pk_cols = {"trips": "trip_id", "bikes": "bike_id", "stations": "station_id"}
    if name in pk_cols:
        pk = pk_cols[name]
        duplicated_count = df.duplicated(subset=[pk]).sum()
        print(f"\nDuplicates based on key '{pk}': {duplicated_count}")

    # Value counts or basic stats for important categorical columns
    if name == "bikes":
        print("\nBike types:")
        print(df["bike_type"].value_counts(dropna=False))
    elif name == "stations":
        print("\nZones:")
        print(df["zone"].value_counts(dropna=False))
    elif name == "trips":
        print("\nDuration Statistics:")
        print(df["duration_min"].describe())
        print("\nUser types:")
        print(df["user_type"].value_counts(dropna=False))
    print("\n" + "="*50 + "\n")

=== Table: trips (612 rows, 7 columns) ===


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member



--- Info & Nulls ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   trip_id           612 non-null    object 
 1   bike_id           612 non-null    object 
 2   start_station_id  612 non-null    object 
 3   end_station_id    612 non-null    object 
 4   start_time        612 non-null    object 
 5   duration_min      582 non-null    float64
 6   user_type         612 non-null    object 
dtypes: float64(1), object(6)
memory usage: 33.6+ KB
None

Duplicates based on key 'trip_id': 12

Duration Statistics:
count     582.000000
mean       27.891753
std       145.064305
min         4.000000
25%        12.000000
50%        16.000000
75%        22.000000
max      2880.000000
Name: duration_min, dtype: float64

User types:
user_type
Member     331
Casual     190
member      35
MEMBER      28
CASUAL      11
Casual       9
casual       8
N

,bike_id,bike_type,purchase_year
0,B001,Standard,2021
1,B002,Electric,2022
2,B003,electric,2024
3,B004,Standard,2021
4,B005,STANDARD,2022



--- Info & Nulls ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   bike_id        60 non-null     object
 1   bike_type      60 non-null     object
 2   purchase_year  60 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 1.5+ KB
None

Duplicates based on key 'bike_id': 0

Bike types:
bike_type
Standard     36
Electric      9
STANDARD      8
standard      3
Electric      2
electric      1
E-bike        1
Name: count, dtype: int64


=== Table: stations (12 rows, 4 columns) ===


,station_id,station_name,zone,capacity
0,S01,Market Square,Centre,29.0
1,S02,Railway Station,Centre,23.0
2,S03,Canal Basin,North,24.0
3,S04,University Gate,West,28.0
4,S05,Park Road,South,22.0



--- Info & Nulls ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   station_id    12 non-null     object 
 1   station_name  12 non-null     object 
 2   zone          12 non-null     object 
 3   capacity      11 non-null     float64
dtypes: float64(1), object(3)
memory usage: 516.0+ bytes
None

Duplicates based on key 'station_id': 0

Zones:
zone
Centre    5
North     2
West      2
South     2
East      1
Name: count, dtype: int64




1. What does one row of each table represent?

**trips**: One row represents a single bike hire trip starting at a specific time/station and ending at another station with a duration in minutes.

**bikes**: One row represents an individual physical bicycle in the fleet, identified by its ID, with its type and purchase year.

**stations**: One row represents a docking station with its name, area/zone, and capacity.

2. Key columns we will use:

**trips**: bike_id, start_station_id, and duration_min.

**bikes**: bike_id and bike_type.

**stations**: station_id and zone.

3. Issues, inconsistencies, and oddities discovered:

trips table:

*   Missing Values: There are 30 missing values in duration_min (582 non-null vs 612 rows).
*   Duplicates: There are 12 duplicated trip_id records that need to be removed or investigated.
*   Inconsistent text capitalization: The user_type column has mixed casing (Member, member, MEMBER, Casual, casual, CASUAL).
*   Outliers: The maximum trip duration is 2,880 minutes (48 hours), which is exceptionally long and could distort averages.

bikes table:

* Inconsistent categories: The bike_type column has inconsistent strings representing the same things (Standard, STANDARD, standard, Electric, Electric , electric, E-bike). These must be standardized.

* stations table: Missing capacity: Some stations (like Old Town Hall S09) have a NaN capacity (missing values). Although capacity isn't strictly needed to calculate average duration by zone, it's good to note.

## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

In [5]:
# 2. Clean and prepare
import pandas as pd

# Create copies of our raw dataframes to keep the original raw data intact
cleaned_trips = tables["trips"].copy()
cleaned_bikes = tables["bikes"].copy()
cleaned_stations = tables["stations"].copy()

# --- A. Clean trips table ---
# 1. Remove duplicates
initial_trips_count = len(cleaned_trips)
cleaned_trips = cleaned_trips.drop_duplicates(subset=["trip_id"])
print(f"Removed {initial_trips_count - len(cleaned_trips)} duplicate trips.")

# 2. Filter out missing durations
initial_durations_count = len(cleaned_trips)
cleaned_trips = cleaned_trips.dropna(subset=["duration_min"])
print(f"Removed {initial_durations_count - len(cleaned_trips)} trips due to missing duration.")

# 3. Handle trip outliers (optional, but let's filter out extreme duration values e.g., > 24 hours / 1440 mins if appropriate)
# For this task, we will keep them unless they bias the average too heavily, but let's write clean values

# 4. Standardize user_type (for general cleanliness)
cleaned_trips["user_type"] = cleaned_trips["user_type"].str.strip().str.capitalize()

# --- B. Clean bikes table ---
# Standardize bike_type values to 'Standard' or 'Electric'
def clean_bike_type(btype):
    btype_str = str(btype).strip().lower()
    if "electric" in btype_str or "e-bike" in btype_str:
        return "Electric"
    elif "standard" in btype_str:
        return "Standard"
    return "Unknown"

cleaned_bikes["bike_type"] = cleaned_bikes["bike_type"].apply(clean_bike_type)
print("\nCleaned Bike Types:")
print(cleaned_bikes["bike_type"].value_counts())

# --- C. Register cleaned tables back to DuckDB for section 3 ---
con.register("trips", cleaned_trips)
con.register("bikes", cleaned_bikes)
con.register("stations", cleaned_stations)

# Update our local dictionary reference as well
tables["trips"] = cleaned_trips
tables["bikes"] = cleaned_bikes
tables["stations"] = cleaned_stations

Removed 12 duplicate trips.
Removed 30 trips due to missing duration.

Cleaned Bike Types:
bike_type
Standard    47
Electric    13
Name: count, dtype: int64


### Cleaning Actions and Decisions

1. **Duplicate Trips**:
   - *Problem*: Found 12 duplicate records based on `trip_id`.
   - *Action*: Removed these duplicate rows using `drop_duplicates(subset=['trip_id'])` to avoid double-counting trips.

2. **Missing Durations**:
   - *Problem*: Found 30 trips with missing values in `duration_min`.
   - *Action*: Dropped these rows since we cannot calculate accurate average trip durations if the duration itself is unknown.

3. **Inconsistent Bike Types**:
   - *Problem*: The `bike_type` column had multiple variations (e.g., `STANDARD`, `standard`, `Electric`, `electric`, `E-bike`).
   - *Action*: Standardized all variations into two clean categories: `Standard` and `Electric` using a mapping function.

4. **Inconsistent User Types**:
   - *Problem*: Mixed casing in `user_type` (e.g., `Member`, `member`, `MEMBER`).
   - *Action*: Capitalized and stripped whitespace to standardize them (e.g., `Member` and `Casual`).

## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

In [6]:
# 3. Join and aggregate

# Let's write a SQL query to join our cleaned tables and aggregate the results
query = """
SELECT
    b.bike_type,
    s.zone AS start_zone,
    COUNT(t.trip_id) AS total_trips,
    ROUND(AVG(t.duration_min), 1) AS average_duration_min
FROM trips AS t
JOIN bikes AS b ON t.bike_id = b.bike_id
JOIN stations AS s ON t.start_station_id = s.station_id
GROUP BY b.bike_type, s.zone
ORDER BY b.bike_type, s.zone
"""

# Execute query and store as a final dataframe
final_summary_df = q(query)

# Display the final summary
display(final_summary_df)

,bike_type,start_zone,total_trips,average_duration_min
0,Electric,Centre,81,14.5
1,Electric,East,11,15.7
2,Electric,North,23,16.0
3,Electric,South,24,13.4
4,Electric,West,22,15.7
5,Standard,Centre,217,38.7
6,Standard,East,20,19.5
7,Standard,North,39,20.4
8,Standard,South,60,43.2
9,Standard,West,73,19.8


### Row Representation and Expectations

- **What one row represents**: Each row represents a distinct combination of **bike type** (Standard or Electric) and the **zone of the start station** (e.g., Centre, North, West, South), showing the total number of trips made and their average duration in minutes.
- **Join Expectations**:
  - Cleaned `trips` has **570** rows.
  - Since every `bike_id` in `trips` should match a bike in `bikes`, and every `start_station_id` should match a station in `stations` (all IDs are standardized and present), we expect **570** total trips to be aggregated in our output table.

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

In [7]:
# 4. Check

# Check 1: Do the total trips across all zones and types sum to exactly 570?
sum_trips = final_summary_df["total_trips"].sum()
expected_trips = len(tables["trips"])
print(f"Sum of trips in aggregated table: {sum_trips}")
print(f"Expected trips from cleaned_trips: {expected_trips}")
assert sum_trips == expected_trips, "Mismatch in total trip count!"
print("✔ Total trip counts match perfectly!")

# Check 2: Let's manually calculate stats for Electric bikes starting in the East zone
# to spot-check the averages.
east_electric_trips = tables["trips"].merge(tables["bikes"], on="bike_id").merge(tables["stations"], left_on="start_station_id", right_on="station_id")
subset = east_electric_trips[(east_electric_trips["bike_type"] == "Electric") & (east_electric_trips["zone"] == "East")]

print("\n--- Spot Check (Electric, East Zone) ---")
print(f"Pandas calculated total trips: {len(subset)}")
print(f"Pandas calculated mean duration: {subset['duration_min'].mean():.2f} minutes")

# Get values from our SQL summary dataframe to compare
sql_row = final_summary_df[(final_summary_df["bike_type"] == "Electric") & (final_summary_df["start_zone"] == "East")].iloc[0]
print(f"SQL summary total trips: {sql_row['total_trips']}")
print(f"SQL summary mean duration: {sql_row['average_duration_min']}")

if len(subset) == sql_row['total_trips'] and round(subset['duration_min'].mean(), 1) == sql_row['average_duration_min']:
    print("✔ Spot check passed! The calculations match exactly.")
else:
    print("❌ Spot check failed. Please review the joins.")

Sum of trips in aggregated table: 570
Expected trips from cleaned_trips: 570
✔ Total trip counts match perfectly!

--- Spot Check (Electric, East Zone) ---
Pandas calculated total trips: 11
Pandas calculated mean duration: 15.73 minutes
SQL summary total trips: 11
SQL summary mean duration: 15.7
✔ Spot check passed! The calculations match exactly.


### Verification Analysis

To ensure our answers are accurate, we conducted two checks:
- **Total Row Sum Consistency**: Checked if our final aggregated group sum equals our total cleaned dataset rows (**570**). This verifies no trips were lost or duplicated during the SQL `JOIN` processes.
- **Manual Spot Check**: Isolated a specific slice (`Electric` bikes in the `East` zone) via Pandas directly and compared the mean duration and counts to the SQL-generated summary. The two sources aligned exactly.

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

### Prompts Sent to AI

1. **Look at the data**:
   - *Prompt*: "Look at the data and identify issues like duplicates, missing values, and capitalization inconsistencies."
   - *Decision*: Accepted the plan to print data types, nulls, distinct counts, and categorical value counts.

2. **Clean and prepare**:
   - *Prompt*: "Clean and prepare the tables based on the findings from step 1: remove duplicate trip IDs, drop missing trip durations, and standardize bike types and user types."
   - *Decision*: Accepted and verified that 12 duplicates and 30 missing value rows were successfully cleaned.

3. **Join and aggregate**:
   - *Prompt*: "Write a SQL query in DuckDB to calculate average trip duration and total trips by bike type and start station zone."
   - *Decision*: Fully accepted. Checked that the join output structure matched the expected combinations.

4. **Check**:
   - *Prompt*: "Write verification checks to ensure total trip counts are consistent and manually spot-check one of the categories."
   - *Decision*: Successfully verified and confirmed matching numbers.

## 6. What I could not verify

*What remains unchecked? What would you do next?*

### What Remains Unverified / Future Work

1. **Extremely Long Trips**:
   - We noticed a maximum trip duration of **2,880 minutes (48 hours)** in our summary statistics. We cannot verify if this represents a real customer scenario (e.g., a bike kept over a weekend or a system error where a bike wasn't docked properly). In a real-world scenario, we would ask the manager for a threshold to exclude or cap extremely long trips.

2. **Station Capacity Accuracy**:
   - We observed that some stations (like `S09` - `Old Town Hall`) have `NaN` capacity values. We cannot verify or fill these capacities without external ground-truth records.

3. **Temporal Patterns**:
   - We did not verify whether the relationship between bike types and trip duration differs across different times of day or days of the week, which would be the logical next step for deeper insights.